In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

# Paths
MODEL_PATH = (
    "/content/drive/MyDrive/ASD_XAI_Project/models/"
    "squeezenet_baseline.keras"
)

VAL_PATH = (
    "/content/drive/MyDrive/ASD_XAI_Project/metadata/"
    "val_slices.csv"
)

IG_PATH = (
    "/content/drive/MyDrive/ASD_XAI_Project/"
    "experiment_06_squeezenet_gradcam_ig/"
    "ig_complete_ranking.csv"
)

In [3]:
# Load model
squeezenet_model = tf.keras.models.load_model(
    MODEL_PATH
)

# Load validation metadata
val_slices_df = pd.read_csv(VAL_PATH)

# Load saved IG rankings
ig_ranked_df = pd.read_csv(IG_PATH)

print("SqueezeNet:", squeezenet_model.input_shape)
print("Validation slices:", len(val_slices_df))
print("Validation subjects:", val_slices_df["SUB_ID"].nunique())
print("IG rows:", len(ig_ranked_df))

SqueezeNet: (None, 224, 224, 3)
Validation slices: 1054
Validation subjects: 6
IG rows: 1054


In [5]:
import pandas as pd
import re

VAL_PATH = "/content/drive/MyDrive/ASD_XAI_Project/metadata/val_slices.csv"
IG_PATH = "/content/drive/MyDrive/ASD_XAI_Project/experiment_06_squeezenet_gradcam_ig/ig_complete_ranking.csv"

val_df = pd.read_csv(VAL_PATH)
ig_df = pd.read_csv(IG_PATH)

print("Validation rows:", len(val_df))
print("IG rows:", len(ig_df))

print("\nValidation columns:")
print(val_df.columns.tolist())

print("\nIG columns:")
print(ig_df.columns.tolist())

Validation rows: 1054
IG rows: 1054

Validation columns:
['SUB_ID', 'SLICE_PATH', 'LABEL', 'SPLIT']

IG columns:
['SUB_ID', 'SLICE_PATH', 'LABEL', 'IG_SCORE', 'IG_RANK']


In [6]:
def extract_slice_number(path):
    match = re.search(r'slice[_-]?(\d+)', str(path).lower())
    return int(match.group(1)) if match else -1


val_df["SLICE_INDEX"] = val_df["SLICE_PATH"].apply(extract_slice_number)
ig_df["SLICE_INDEX"] = ig_df["SLICE_PATH"].apply(extract_slice_number)

print("Sample slice paths + extracted numbers:\n")

print(
    val_df[["SUB_ID", "SLICE_PATH", "SLICE_INDEX"]]
    .head(20)
    .to_string(index=False)
)

print("\nNumber of failed extractions:",
      (val_df["SLICE_INDEX"] == -1).sum())

Sample slice paths + extracted numbers:

 SUB_ID                                                                SLICE_PATH  SLICE_INDEX
  51487 /content/drive/MyDrive/ASD_XAI_Project/slices/Control/51487/slice_000.jpg            0
  51487 /content/drive/MyDrive/ASD_XAI_Project/slices/Control/51487/slice_001.jpg            1
  51487 /content/drive/MyDrive/ASD_XAI_Project/slices/Control/51487/slice_002.jpg            2
  51487 /content/drive/MyDrive/ASD_XAI_Project/slices/Control/51487/slice_003.jpg            3
  51487 /content/drive/MyDrive/ASD_XAI_Project/slices/Control/51487/slice_004.jpg            4
  51487 /content/drive/MyDrive/ASD_XAI_Project/slices/Control/51487/slice_005.jpg            5
  51487 /content/drive/MyDrive/ASD_XAI_Project/slices/Control/51487/slice_006.jpg            6
  51487 /content/drive/MyDrive/ASD_XAI_Project/slices/Control/51487/slice_007.jpg            7
  51487 /content/drive/MyDrive/ASD_XAI_Project/slices/Control/51487/slice_008.jpg            8
  51487 /

In [7]:
import numpy as np

# Merge IG scores with the validation metadata
regional_df = ig_df.copy()

# Make sure slices are in anatomical/numerical order
regional_df = regional_df.sort_values(
    ["SUB_ID", "SLICE_INDEX"]
).reset_index(drop=True)

In [8]:
NUM_REGIONS = 10

selected_rows = []

for sub_id, subject_df in regional_df.groupby("SUB_ID"):

    subject_df = subject_df.sort_values("SLICE_INDEX").reset_index(drop=True)

    n_slices = len(subject_df)

    # Divide the subject's slices into 10 approximately equal regions
    region_edges = np.linspace(0, n_slices, NUM_REGIONS + 1).astype(int)

    for region in range(NUM_REGIONS):

        start = region_edges[region]
        end = region_edges[region + 1]

        region_df = subject_df.iloc[start:end]

        if len(region_df) == 0:
            continue

        # Pick the highest IG slice in this region
        best_slice = region_df.loc[
            region_df["IG_SCORE"].idxmax()
        ].copy()

        best_slice["REGION"] = region + 1

        selected_rows.append(best_slice)


regional_selected_df = pd.DataFrame(selected_rows)

print("Selected slices:", len(regional_selected_df))
print("Subjects:", regional_selected_df["SUB_ID"].nunique())

print("\nSlices selected per subject:")
print(
    regional_selected_df.groupby("SUB_ID")
    .size()
)

print("\nSelected slices:")
print(
    regional_selected_df[
        ["SUB_ID", "SLICE_INDEX", "IG_SCORE", "IG_RANK", "REGION"]
    ].to_string(index=False)
)

Selected slices: 60
Subjects: 6

Slices selected per subject:
SUB_ID
51460    10
51463    10
51466    10
51475    10
51476    10
51487    10
dtype: int64

Selected slices:
 SUB_ID  SLICE_INDEX     IG_SCORE  IG_RANK  REGION
  51460           16 3.013414e-07    114.0       1
  51460           26 9.728561e-07     18.0       2
  51460           35 9.233231e-07     24.0       3
  51460           52 6.700868e-07     61.0       4
  51460           70 3.565194e-07     93.0       5
  51460           91 2.786356e-07    118.0       6
  51460          120 5.905500e-07     65.0       7
  51460          139 1.128797e-06     10.0       8
  51460          148 1.291659e-06      1.0       9
  51460          158 1.222618e-07    144.0      10
  51463           16 2.332977e-09    148.0       1
  51463           31 7.292763e-07      1.0       2
  51463           35 6.688220e-07      6.0       3
  51463           54 3.185554e-07     56.0       4
  51463           79 1.440304e-07     84.0       5
  51463     

In [9]:
SAVE_PATH = "/content/drive/MyDrive/ASD_XAI_Project/experiment_07_ig_regional_selection"

import os
os.makedirs(SAVE_PATH, exist_ok=True)

regional_selected_df.to_csv(
    f"{SAVE_PATH}/regional_ig_selected_slices.csv",
    index=False
)

print("Saved successfully!")
print(f"Path: {SAVE_PATH}/regional_ig_selected_slices.csv")

Saved successfully!
Path: /content/drive/MyDrive/ASD_XAI_Project/experiment_07_ig_regional_selection/regional_ig_selected_slices.csv


In [11]:
from tensorflow.keras.models import load_model

MODEL_PATH = "/content/drive/MyDrive/ASD_XAI_Project/models/squeezenet_baseline.keras"

model = load_model(MODEL_PATH)

print("SqueezeNet loaded successfully!")
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)

SqueezeNet loaded successfully!
Input shape: (None, 224, 224, 3)
Output shape: (None, 1)


In [12]:
# Load the saved regional selection
selected_df = pd.read_csv(
    f"{SAVE_PATH}/regional_ig_selected_slices.csv"
)

print("Selected slices:", len(selected_df))


def load_slice(img_path):
    img = image.load_img(
        img_path,
        target_size=(224, 224)
    )

    img = image.img_to_array(img)
    img = img / 255.0

    return img


# Load selected images
X_selected = np.array([
    load_slice(path)
    for path in selected_df["SLICE_PATH"]
])

print("X_selected shape:", X_selected.shape)


# Predict using trained SqueezeNet
pred_probs = model.predict(
    X_selected,
    batch_size=16,
    verbose=1
).flatten()

selected_df["PRED_PROB"] = pred_probs

print("\nPrediction complete!")
print(selected_df[
    ["SUB_ID", "SLICE_INDEX", "REGION", "IG_SCORE", "PRED_PROB"]
].head(20))

Selected slices: 60
X_selected shape: (60, 224, 224, 3)
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 357ms/step

Prediction complete!
    SUB_ID  SLICE_INDEX  REGION      IG_SCORE  PRED_PROB
0    51460           16       1  3.013414e-07   0.522859
1    51460           26       2  9.728561e-07   0.522859
2    51460           35       3  9.233231e-07   0.522859
3    51460           52       4  6.700868e-07   0.522859
4    51460           70       5  3.565194e-07   0.522859
5    51460           91       6  2.786356e-07   0.522859
6    51460          120       7  5.905500e-07   0.522859
7    51460          139       8  1.128797e-06   0.522859
8    51460          148       9  1.291659e-06   0.522859
9    51460          158      10  1.222618e-07   0.522859
10   51463           16       1  2.332977e-09   0.522859
11   51463           31       2  7.292763e-07   0.522859
12   51463           35       3  6.688220e-07   0.522859
13   51463           54       4  3.185554e-07   0.522859
14   51463           79     

In [13]:
print("X_selected shape:", X_selected.shape)

print("\nPixel statistics:")
print("Slice 0  - min:", X_selected[0].min(),
      "max:", X_selected[0].max(),
      "mean:", X_selected[0].mean())

print("Slice 10 - min:", X_selected[10].min(),
      "max:", X_selected[10].max(),
      "mean:", X_selected[10].mean())

print("Slice 30 - min:", X_selected[30].min(),
      "max:", X_selected[30].max(),
      "mean:", X_selected[30].mean())

print("Slice 59 - min:", X_selected[59].min(),
      "max:", X_selected[59].max(),
      "mean:", X_selected[59].mean())

# Check whether the images are actually identical
print("\nAre slice 0 and slice 10 identical?",
      np.array_equal(X_selected[0], X_selected[10]))

print("Are slice 0 and slice 59 identical?",
      np.array_equal(X_selected[0], X_selected[59]))

X_selected shape: (60, 224, 224, 3)

Pixel statistics:
Slice 0  - min: 0.0 max: 0.4392157 mean: 0.03460908
Slice 10 - min: 0.003921569 max: 0.3647059 mean: 0.022273831
Slice 30 - min: 0.0 max: 0.43529412 mean: 0.03775425
Slice 59 - min: 0.0 max: 0.43529412 mean: 0.04367732

Are slice 0 and slice 10 identical? False
Are slice 0 and slice 59 identical? False


In [14]:
# Test the model on completely different selected slices
test_indices = [0, 1, 10, 20, 30, 40, 50, 59]

test_probs = model.predict(
    X_selected[test_indices],
    verbose=0
).flatten()

print("Model predictions:")
for idx, prob in zip(test_indices, test_probs):
    print(f"Slice {idx:2d}: {prob:.12f}")

print("\nUnique prediction values:",
      len(np.unique(np.round(test_probs, 12))))

Model predictions:
Slice  0: 0.522858619690
Slice  1: 0.522858679295
Slice 10: 0.522858619690
Slice 20: 0.522858679295
Slice 30: 0.522858679295
Slice 40: 0.522858679295
Slice 50: 0.522858679295
Slice 59: 0.522858679295

Unique prediction values: 2


In [15]:
IMG_SIZE = 224

def load_image(path):
    image = tf.io.read_file(path)

    image = tf.image.decode_png(
        image,
        channels=3
    )

    image = tf.image.resize(
        image,
        (IMG_SIZE, IMG_SIZE)
    )

    image = tf.cast(
        image,
        tf.float32
    )

    return image

In [16]:
import numpy as np
import pandas as pd

selected_df = pd.read_csv(
    "/content/drive/MyDrive/ASD_XAI_Project/"
    "experiment_07_ig_regional_selection/"
    "regional_ig_selected_slices.csv"
)

subject_results = []

for sub_id, group in selected_df.groupby("SUB_ID"):

    # IMPORTANT:
    # Use the EXACT load_image() from Experiment 06
    images = np.stack([
        load_image(path).numpy()
        for path in group["SLICE_PATH"]
    ])

    predictions = model.predict(
        images,
        batch_size=16,
        verbose=0
    ).reshape(-1)

    # EXACT Experiment 06 aggregation
    probability = np.mean(predictions)

    # EXACT Experiment 06 threshold
    predicted_label = (
        "ASD"
        if probability >= 0.5
        else "Control"
    )

    true_label = group["LABEL"].iloc[0]

    subject_results.append({
        "SUB_ID": sub_id,
        "TRUE_LABEL": true_label,
        "PREDICTED_LABEL": predicted_label,
        "PROBABILITY": probability,
        "NUM_SLICES": len(group)
    })

subject_df = pd.DataFrame(subject_results)

accuracy = (
    subject_df["TRUE_LABEL"]
    == subject_df["PREDICTED_LABEL"]
).mean()

print("=== EXPERIMENT 07 RESULTS ===")
print(subject_df)

print("\nAccuracy:", accuracy)
print("Correct:", int(accuracy * len(subject_df)), "/", len(subject_df))

=== EXPERIMENT 07 RESULTS ===
   SUB_ID TRUE_LABEL PREDICTED_LABEL  PROBABILITY  NUM_SLICES
0   51460        ASD         Control     0.484490          10
1   51463        ASD             ASD     0.501792          10
2   51466        ASD         Control     0.496756          10
3   51475    Control         Control     0.479290          10
4   51476    Control         Control     0.476613          10
5   51487    Control         Control     0.491508          10

Accuracy: 0.6666666666666666
Correct: 4 / 6


In [17]:
# ============================================================
# SAVE EXPERIMENT 07 FINAL RESULTS
# ============================================================

EXP7_DIR = "/content/drive/MyDrive/ASD_XAI_Project/experiment_07_ig_regional_selection"

# 1. Save subject-level prediction results
subject_df.to_csv(
    f"{EXP7_DIR}/regional_ig_subject_results.csv",
    index=False
)

# 2. Save experiment summary
summary_df = pd.DataFrame([{
    "EXPERIMENT": "Experiment 07",
    "MODEL": "SqueezeNet",
    "XAI_METHOD": "Integrated Gradients",
    "SELECTION_METHOD": "Regional / Diverse Selection",
    "REGIONS_PER_SUBJECT": 10,
    "SLICES_PER_SUBJECT": 10,
    "TOTAL_SELECTED_SLICES": len(selected_df),
    "TOTAL_VALIDATION_SLICES": len(val_df),
    "AGGREGATION": "Mean",
    "THRESHOLD": 0.5,
    "CORRECT_SUBJECTS": int(
        (subject_df["TRUE_LABEL"] == subject_df["PREDICTED_LABEL"]).sum()
    ),
    "TOTAL_SUBJECTS": len(subject_df),
    "ACCURACY": accuracy
}])

summary_df.to_csv(
    f"{EXP7_DIR}/experiment_07_summary.csv",
    index=False
)

print("✅ Experiment 07 results saved successfully!")
print()
print("Subject results:")
display(subject_df)

print("\nExperiment summary:")
display(summary_df)

✅ Experiment 07 results saved successfully!

Subject results:


,SUB_ID,TRUE_LABEL,PREDICTED_LABEL,PROBABILITY,NUM_SLICES
0,51460,ASD,Control,0.484490,10
1,51463,ASD,ASD,0.501792,10
2,51466,ASD,Control,0.496756,10
3,51475,Control,Control,0.479290,10
4,51476,Control,Control,0.476613,10
5,51487,Control,Control,0.491508,10



Experiment summary:


,EXPERIMENT,MODEL,XAI_METHOD,SELECTION_METHOD,REGIONS_PER_SUBJECT,SLICES_PER_SUBJECT,TOTAL_SELECTED_SLICES,TOTAL_VALIDATION_SLICES,AGGREGATION,THRESHOLD,CORRECT_SUBJECTS,TOTAL_SUBJECTS,ACCURACY
0,Experiment 07,SqueezeNet,Integrated Gradients,Regional / Diverse Selection,10,10,60,1054,Mean,0.5,4,6,0.666667
